# Paradigmas de Programação — Aula 1
## Notebook 1 de 3 — Um problema, quatro estilos

**Universidade La Salle · Disciplina: Paradigmas de Programação**

**Objetivo:** resolver exatamente o mesmo problema em quatro estilos de programação e medir, para cada um, linhas de código, tempo de execução e número de mutações de estado. A conclusão da aula não é qual estilo é *melhor*, e sim que cada um torna barata uma coisa diferente.

> Como usar: menu *Ambiente de execução → Executar tudo* (`Ctrl+F9`). Nada precisa ser instalado — o Colab já traz `numpy`, `pandas` e `matplotlib`.


In [ ]:
# Configuração comum a todos os notebooks da disciplina
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# Paleta da disciplina: um paradigma, uma cor
CORES = {
    "Imperativo": "#B85042",
    "Funcional":  "#2C5F2D",
    "Orientado a objetos": "#065A82",
    "Lógico":     "#6D2E46",
    "Concorrente": "#C77D00",
    "Declarativo": "#50808E",
}
print("Ambiente pronto. matplotlib", matplotlib.__version__, "| numpy", np.__version__)


---
## 1. O problema

> Dada uma lista de inteiros, some os **quadrados dos números pares**.

É deliberadamente pequeno: assim a diferença que aparece nos números vem do **estilo**, e não da
dificuldade do problema.


In [ ]:
from typing import List

DADOS = list(range(1, 200_001))   # 200 mil inteiros
ESPERADO = sum(x * x for x in DADOS if x % 2 == 0)
print("resultado esperado:", ESPERADO)


---
## 2. Estilo imperativo

Descreve **como** a máquina deve proceder: uma variável acumuladora, um laço, uma mutação por
iteração. É o estilo herdado da arquitetura de von Neumann.


In [ ]:
def soma_imperativa(nums: List[int]) -> int:
    total = 0                      # estado nomeado
    for x in nums:                 # controle explícito
        if x % 2 == 0:
            total = total + x * x  # mutação: 100.000 atribuições
    return total

print(soma_imperativa(DADOS) == ESPERADO)


---
## 3. Estilo funcional

Descreve **o que** é o resultado, como composição de transformações. Não há variável mutável:
`filter` seleciona, `map` transforma, `reduce` colapsa.


In [ ]:
from functools import reduce
from operator import add

def soma_funcional(nums: List[int]) -> int:
    return reduce(add, map(lambda x: x * x, filter(lambda x: x % 2 == 0, nums)), 0)

print(soma_funcional(DADOS) == ESPERADO)

# Versão idiomática em Python, que é funcional no espírito e preguiçosa:
def soma_funcional_py(nums: List[int]) -> int:
    return sum(x * x for x in nums if x % 2 == 0)

print(soma_funcional_py(DADOS) == ESPERADO)


---
## 4. Estilo orientado a objetos

O cálculo vira um **objeto** que guarda estado interno e responde a mensagens. Repare que o estado
mutável continua lá — apenas mudou de lugar, para dentro da cápsula.


In [ ]:
class SomadorDeQuadradosPares:
    def __init__(self):
        self._total = 0

    def oferecer(self, x: int) -> "SomadorDeQuadradosPares":
        if x % 2 == 0:
            self._total += x * x
        return self                 # permite encadeamento

    @property
    def total(self) -> int:
        return self._total

def soma_oo(nums: List[int]) -> int:
    s = SomadorDeQuadradosPares()
    for x in nums:
        s.oferecer(x)
    return s.total

print(soma_oo(DADOS) == ESPERADO)


---
## 5. Estilo declarativo / orientado a arrays

Aqui não há laço nenhum no código Python: a iteração acontece dentro da biblioteca, em C
vetorizado. É o mesmo espírito de SQL — você declara o conjunto, não o percurso.


In [ ]:
ARR = np.arange(1, 200_001, dtype=np.int64)

def soma_vetorizada(arr=ARR) -> int:
    return int((arr[arr % 2 == 0] ** 2).sum())

print(soma_vetorizada() == ESPERADO)


---
## 6. Medindo os quatro estilos

Três métricas: **tempo** (mediana de 5 execuções), **linhas de código efetivas** e **número de
mutações de estado** — quantas vezes uma variável já existente foi sobrescrita.


In [ ]:
import timeit, inspect

# Linhas contadas manualmente, usadas caso inspect.getsource falhe no ambiente
LINHAS_FALLBACK = {"Imperativo": 5, "Funcional": 1, "Orientado a objetos": 14, "Declarativo": 1}

def loc(objetos) -> int:
    """Linhas de código efetivas: ignora linhas em branco e comentários."""
    total = 0
    for obj in objetos:
        try:
            fonte = inspect.getsource(obj)
        except (OSError, TypeError):
            return 0
        total += sum(1 for l in fonte.splitlines()[1:]
                     if l.strip() and not l.strip().startswith("#"))
    return total

MUTACOES_LACO = sum(1 for x in DADOS if x % 2 == 0)   # uma atribuição por número par

# (nome, função para contar linhas, chamada sem argumentos, mutações de estado)
IMPLEMENTACOES = [
    ("Imperativo",          [soma_imperativa],   lambda: soma_imperativa(DADOS),   MUTACOES_LACO),
    ("Funcional",           [soma_funcional_py], lambda: soma_funcional_py(DADOS), 0),
    # o estilo OO precisa da classe e da função: as duas contam como código escrito
    ("Orientado a objetos", [SomadorDeQuadradosPares, soma_oo], lambda: soma_oo(DADOS), MUTACOES_LACO),
    ("Declarativo",         [soma_vetorizada],   lambda: soma_vetorizada(ARR),     0),
]

resultados = []
for nome, fontes, chamada, mutacoes in IMPLEMENTACOES:
    t = min(timeit.repeat(chamada, repeat=5, number=1))
    linhas = loc(fontes) or LINHAS_FALLBACK[nome]
    resultados.append({"estilo": nome, "tempo_ms": t * 1000,
                       "linhas": linhas, "mutacoes": mutacoes})
    print(f"{nome:22s} {t*1000:8.2f} ms  |  {linhas} linhas  |  {mutacoes} mutações")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
estilos = [r["estilo"] for r in resultados]
cores = [CORES[e] for e in estilos]
rotulos = [e.replace(" a ", "\na ") for e in estilos]

series = [("tempo_ms", "Tempo de execução (ms)", "{:.1f}"),
          ("linhas",   "Linhas de código efetivas", "{:.0f}"),
          ("mutacoes", "Mutações de estado", "{:,.0f}")]

for ax, (chave, titulo, fmt) in zip(axes, series):
    valores = [r[chave] for r in resultados]
    barras = ax.bar(rotulos, valores, color=cores)
    ax.set_title(titulo, fontsize=11, weight="bold")
    ax.tick_params(axis="x", labelsize=9)
    ax.bar_label(barras, labels=[fmt.format(v) for v in valores], fontsize=9, padding=2)
    ax.set_ylim(0, max(valores) * 1.25 if max(valores) > 0 else 1)

fig.suptitle("O mesmo problema em quatro estilos — o que cada um torna barato",
             fontsize=13, weight="bold")
fig.tight_layout()
plt.show()


### Como ler o gráfico

- **Tempo** — o estilo declarativo vence porque delega o laço a código vetorizado; o funcional
  puro com `reduce`/`lambda` costuma ser o mais lento em Python, por causa do custo de chamada.
- **Linhas** — a contagem cai conforme subimos o nível de abstração. Menos linhas significa menos
  lugares onde errar, mas também menos controle explícito.
- **Mutações** — esta é a coluna conceitualmente decisiva. O imperativo e o OO fazem 100 mil
  mutações; o funcional e o declarativo, nenhuma. **Zero mutação é o que torna um trecho
  trivialmente paralelizável e trivialmente testável** — vamos voltar a isso nas Aulas 4 e 14.

> Cuidado com a leitura fácil: *rápido em Python* não é *rápido em geral*. Em Haskell ou Elixir a
> ordem da primeira coluna muda. O que **não** muda entre linguagens é a terceira coluna.


---
## 7. Exercícios

1. Acrescente uma quinta implementação em estilo **recursivo puro** (sem laço e sem acumulador
   mutável) e inclua-a nas três medições. O que acontece com `DADOS` de 200 mil elementos? Por quê?
2. Reduza `DADOS` para 1.000 elementos e refaça os gráficos. A ordem das barras de tempo muda?
   O que isso diz sobre benchmarks pequenos?
3. Reescreva `soma_oo` de modo que o objeto seja **imutável** (cada `oferecer` devolve um objeto
   novo). Meça de novo e explique o resultado.
